# Pneumonia Detection - Custom CNN with Performance Optimizations
This notebook trains a custom CNN to detect pneumonia from chest X-ray images with improved architecture and techniques to exceed 90% accuracy.

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from datetime import datetime
import pickle

# =============================================================================
# CONFIGURATION
# =============================================================================
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
INITIAL_LR = 0.0001
PATIENCE_EARLY = 10
PATIENCE_LR = 5


BASE_DIR = 'chest_xray'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR   = os.path.join(BASE_DIR, 'val')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

# Output directories
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

# =============================================================================
# DATA GENERATORS (with augmentation for training)
# =============================================================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['NORMAL', 'PNEUMONIA'],
    shuffle=True,
    seed=42
)

validation_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['NORMAL', 'PNEUMONIA'],
    shuffle=False,
    seed=42
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    classes=['NORMAL', 'PNEUMONIA'],
    shuffle=False,
    seed=42
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")

# =============================================================================
# BUILD CNN ARCHITECTURE 
# =============================================================================
model = Sequential([
    # Block 1
    Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(IMG_SIZE,IMG_SIZE,3)),
    MaxPooling2D(2,2),

    # Block 2
    Conv2D(64, (3,3), padding='same', activation='relu'),
    MaxPooling2D(2,2),

    # Block 3 & 4
    Conv2D(128, (3,3), padding='same', activation='relu'),
    Conv2D(128, (3,3), padding='same', activation='relu'),
    MaxPooling2D(2,2),

    # Block 5 & 6
    Conv2D(256, (3,3), padding='same', activation='relu'),
    Conv2D(256, (3,3), padding='same', activation='relu'),
    MaxPooling2D(2,2),

    # Block 7
    Conv2D(512, (3,3), padding='same', activation='relu'),

    # Global Average Pooling 
    GlobalAveragePooling2D(),

    # Dropout and Dense layers
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR),
    loss='binary_crossentropy',
    metrics=['accuracy']          # Accuracy as sole metric
)

model.summary()

# =============================================================================
# CALLBACKS
# =============================================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_path = f'models/pneumonia_cnn_best_{timestamp}.h5'


callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=PATIENCE_EARLY,
                  restore_best_weights=True, mode='max', verbose=1),

    ModelCheckpoint(checkpoint_path, monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),

    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                      patience=PATIENCE_LR, mode='max', verbose=1)
]


# =============================================================================
# TRAINING
# =============================================================================
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# =============================================================================
# EVALUATION ON TEST SET
# =============================================================================
# Load best model saved during training
best_model = tf.keras.models.load_model(checkpoint_path)

# Reset test generator
test_generator.reset()
predictions = best_model.predict(test_generator, verbose=1)
predicted_classes = (predictions > 0.5).astype(int).flatten()
true_classes = test_generator.classes

# Accuracy
test_acc = accuracy_score(true_classes, predicted_classes)
print(f"\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# Confusion Matrix
cm = confusion_matrix(true_classes, predicted_classes)
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix:")
print(cm)

# Classification report
print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes,
                            target_names=['NORMAL', 'PNEUMONIA']))

# =============================================================================
# PLOT ACCURACY CURVES
# =============================================================================
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(f'results/accuracy_curves_{timestamp}.png')
plt.show()

# =============================================================================
# PLOT CONFUSION MATRIX
# =============================================================================
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NORMAL','PNEUMONIA'],
            yticklabels=['NORMAL','PNEUMONIA'])
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig(f'results/confusion_matrix_{timestamp}.png')
plt.show()

# =============================================================================
# SAVE RESULTS AND COMPARE WITH BENCHMARKS
# =============================================================================
# Benchmark accuracies from literature
benchmarks = {
    'Kermany et al. (2018)': 0.9280,
    'Stephen et al. (2019)': 0.9300,
    'Becker (2022)': 0.9000   # >90% reported
}

report = f"""
=========================================================================
                    PNEUMONIA DETECTION CNN
                    ACCURACY PERFORMANCE REPORT
=========================================================================
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: Custom CNN (7 conv layers, GlobalAvgPool)

Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)

Confusion Matrix Breakdown:
  True Positives (Pneumonia correct): {tp}
  True Negatives (Normal correct):    {tn}
  False Positives (Normal as Pneumonia): {fp}
  False Negatives (Pneumonia as Normal): {fn}

Comparison with Benchmarks (Accuracy):
"""
for name, acc in benchmarks.items():
    comparison = "higher" if test_acc > acc else "lower" if test_acc < acc else "equal"
    report += f"  {name}: {acc:.4f} -> Our model is {comparison}\n"

report += f"""
=========================================================================
Model Parameters: {best_model.count_params():,}
Model File: {checkpoint_path}
=========================================================================
"""

print(report)
with open(f'results/accuracy_report_{timestamp}.txt', 'w') as f:
    f.write(report)

# =============================================================================
# SAVE TRAINING HISTORY
# =============================================================================
with open(f'results/training_history_{timestamp}.pkl', 'wb') as f:
    pickle.dump(history.history, f)

print("\n✅ Training completed. All results saved in 'results/' folder.")

c:\Users\janet taguta\Desktop\PROJECT\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


GPUs available: 0
Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Training samples: 5216
Validation samples: 16
Test samples: 624

Class weights for imbalance handling:
  NORMAL: 1.945
  PNEUMONIA: 0.673

BUILDING OPTIMIZED CUSTOM CNN


c:\Users\janet taguta\Desktop\PROJECT\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 112, 112, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 56, 56, 256)    │             

 Total params: 5,123,649 (19.55 MB)

 Trainable params: 5,118,017 (19.52 MB)

 Non-trainable params: 5,632 (22.00 KB)


✅ Optimized Custom CNN Created!

TRAINING OPTIMIZED CUSTOM CNN
Epoch 1/50
